In [ ]:
!pip install -q -U transformers accelerate torch

In [ ]:
!pip install -U mistral-common

In [8]:
!pip install -q -U laya

In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

2.10.0+cu128
True
Tesla T4


In [2]:
import transformers

print("Transformers:", transformers.__version__)

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

Transformers: 5.0.0


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [25]:
from laya import Router

laya_router = Router(
    device="cuda",
    default="english"
)

In [26]:
# Testing the raw query (without ADHD brain):
prompt = "Ways to grow my instagram followers from 1K to 10K in a week?"

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.7,
    do_sample=True
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

<think>
Okay, the user wants to grow their Instagram followers from 1K to 10K in a week. That's a huge jump, and I need to think about what's realistic and possible. First, I should consider the time frame. A week is just seven days, so the strategies have to be intense and effective. But I also need to make sure that the advice is ethical and doesn't involve spamming or other black-hat tactics that could get the account banned.

Let me start by breaking down the possible methods. Engagement is key because Instagram's algorithm favors content that gets likes, comments, and shares. So, the user needs to focus on creating high-quality, engaging content. But how to do that quickly? Maybe they can post consistently, using the right hashtags and engaging with others' posts.

Another angle is using Instagram's features like Stories, Reels, and IGTV. Stories have a higher chance of being seen because they're time-sensitive, but they also fade after 24 hours. Reels are more discoverable and ca

In [27]:
state = {
    "problem": "",
    "ideas": [],
    "connections": [],
    "perspectives": [],
    "attention_perturbations": [],
    "cross_connections": [],
    "selected_candidates": [],
    "missing_directions": [],
    "active_threads": [],
    "discarded_ideas": [],
    "active_memory": [],
    "round": 0
}

In [28]:
state["problem"] = "Ways to grow my instagram followers from 1K to 10K in a week?"

In [29]:
import json

def diverge(state):

    prompt = f"""
        Problem:
        {state["problem"]}

        Missing directions from previous round (if running for >1 rounds):
        {state["missing_directions"]}
        
        Generate 10 different ideas for solving this problem.
        
        If missing directions from a previous round are provided,
        use them to deliberately explore areas that were not sufficiently
        covered before.
        
        Do not simply repeat ideas from previous rounds.        
        
        The ideas should be meaningfully different from each other.
        Explore different approaches rather than simply rephrasing the same solution.
        
        Return ONLY valid JSON in exactly this format:
        
        {{
            "ideas": [
                "idea 1",
                "idea 2",
                "idea 3",
                "idea 4",
                "idea 5",
                "idea 6",
                "idea 7",
                "idea 8",
                "idea 9",
                "idea 10"
            ]
        }}
    """
    

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    
    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.8,
        do_sample=True
    )

    
    # response = tokenizer.decode(
    #     outputs[0][inputs["input_ids"].shape[1]:],
    #     skip_special_tokens=True
    # )

    
    # result = json.loads(response)

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    
    print("RAW RESPONSE:")
    print(response)
    
    result = json.loads(response)

    
    state["ideas"].extend(result["ideas"])

    
    return result["ideas"]

In [30]:
# Associative search
def associative_search(state):

    prompt = f"""
Problem:
{state["problem"]}

Ideas discovered so far:
{state["ideas"]}

For each idea, find useful concepts, mechanisms, analogies,
principles, or applications that could lead to new ideas.

Look beyond obvious connections.

Do not simply rephrase the existing ideas.

Return ONLY valid JSON in exactly this format:

{{
    "connections": [
        "connection 1",
        "connection 2",
        "connection 3",
        "connection 4",
        "connection 5",
        "connection 6",
        "connection 7",
        "connection 8",
        "connection 9",
        "connection 10"
    ]
}}
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.8,
        do_sample=True
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    result = json.loads(response)

    state["connections"].extend(result["connections"])

    return result["connections"]

In [31]:
def perspective_switch(state):

    prompt = f"""
        Problem:
        {state["problem"]}
        
        Ideas:
        {state["ideas"]}
        
        Now deliberately change the perspective used to think about this problem.
        
        Explore the problem from different perspectives such as:
        - different users
        - different industries
        - different scales
        - different constraints
        - different goals
        - different disciplines
        
        Generate 10 genuinely different perspectives that could reveal
        new opportunities, problems, or solutions.
        
        Do not simply rephrase the existing ideas.
        
        Return ONLY valid JSON in exactly this format:
        
        {{
            "perspectives": [
                "perspective 1",
                "perspective 2",
                "perspective 3",
                "perspective 4",
                "perspective 5",
                "perspective 6",
                "perspective 7",
                "perspective 8",
                "perspective 9",
                "perspective 10"
            ]
        }}
    """

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.8,
        do_sample=True
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    result = json.loads(response)

    state["perspectives"].extend(result["perspectives"])

    return result["perspectives"]

In [32]:
def attention_perturb(state):

    prompt = f"""
        Problem:
        {state["problem"]}

        Ideas:
        {state["ideas"]}

        Now deliberately attention_perturb the current reasoning.

        Introduce unexpected changes such as:
        - reversing an assumption
        - removing an important constraint
        - adding an extreme constraint
        - combining unrelated elements
        - changing the scale
        - changing the objective
        - imagining the opposite solution

        Generate 10 useful attention_perturbations that could expose
        new ideas or directions.

        Do not simply repeat the existing reasoning.

        Return ONLY valid JSON in exactly this format:

        {{
            "attention_perturbations": [
                "attention_perturbation 1",
                "attention_perturbation 2",
                "attention_perturbation 3",
                "attention_perturbation 4",
                "attention_perturbation 5",
                "attention_perturbation 6",
                "attention_perturbation 7",
                "attention_perturbation 8",
                "attention_perturbation 9",
                "attention_perturbation 10"
            ]
        }}
    """

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.9,
        do_sample=True
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    result = json.loads(response)

    state["attention_perturbations"].extend(result["attention_perturbations"])

    return result["attention_perturbations"]

In [33]:
def cross_connect(state):

    prompt = f"""
        Problem:
        {state["problem"]}
        
        Ideas:
        {state["ideas"]}
        
        Associative connections:
        {state["connections"]}
        
        Perspectives:
        {state["perspectives"]}
        
        attention_perturbations:
        {state["attention_perturbations"]}
        
        Find meaningful connections between different elements
        of the current reasoning.
        
        Look for:
        
        - combinations that could create a new solution
        - unexpected relationships
        - complementary ideas
        - ideas that could solve each other's weaknesses
        - mechanisms from one idea that could improve another
        - connections between different perspectives or attention_perturbations
        
        Generate 10 novel cross-connections.
        
        Do not simply repeat or rephrase existing ideas.
        
        Return ONLY valid JSON in exactly this format:
        
        {{
            "cross_connections": [
                "connection 1",
                "connection 2",
                "connection 3",
                "connection 4",
                "connection 5",
                "connection 6",
                "connection 7",
                "connection 8",
                "connection 9",
                "connection 10"
            ]
        }}
    """

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.9,
        do_sample=True
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    result = json.loads(response)

    state["cross_connections"].extend(result["cross_connections"])

    return result["cross_connections"]

In [34]:
def laya_select(state):

    candidates = state["cross_connections"]

    selected_candidates = []
    decisions = []

    for i, candidate in enumerate(candidates):

        laya_state = {
            "problem": state["problem"],
            "candidate": candidate
        }

        questions = {
            "keep": {
                "type": "noul",
                "instructions": (
                    "Is this candidate worth keeping as a "
                    "promising solution to the original problem?"
                ),
                "criteria": {
                    "true": (
                        "The candidate is relevant, useful, "
                        "meaningfully different, feasible, and "
                        "has strong reasoning."
                    ),
                    "false": (
                        "The candidate is weak, redundant, "
                        "irrelevant, impractical, or poorly reasoned."
                    )
                }
            }
        }

        result = laya_router.predict(
            laya_state,
            questions,
            model="english"
        )

        probability = result["answers"]["keep"]["noul"]

        keep = probability >= 0.5

        decisions.append({
            "candidate": candidate,
            "probability": probability,
            "keep": keep
        })

        if keep:
            selected_candidates.append(candidate)

    state["selected_candidates"] = selected_candidates

    return decisions

In [35]:
def evaluate_similar_or_not(state):

    selected = state["selected_candidates"]

    prompt = f"""
        Problem:
        {state["problem"]}
    
        Selected candidates:
        {selected}
    
        Analyze the selected candidates.
    
        Determine whether the candidates are still too similar in
        their underlying approach or whether important solution
        directions are still missing.
    
        Do NOT judge which candidate is best.
    
        Your task is specifically to evaluate the DIVERSITY and
        COVERAGE of the selected candidates.
    
        If the candidates are sufficiently diverse and cover
        meaningfully different directions, return:
    
        {{
            "still_similar": false,
            "missing_directions": []
        }}
    
        If the candidates are still too similar or important
        directions are missing, return:
    
        {{
            "still_similar": true,
            "missing_directions": [
                "missing direction 1",
                "missing direction 2",
                "missing direction 3"
            ]
        }}
    
        Return ONLY valid JSON.
    """

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.3,
        do_sample=True
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    result = json.loads(response)

    state["missing_directions"] = result["missing_directions"]

    return result

In [36]:
def converge(state):

    prompt = f"""
        Problem:
        {state["problem"]}
        
        Ideas:
        {state["ideas"]}
        
        Connections:
        {state["connections"]}
        
        Perspectives:
        {state["perspectives"]}
        
        attention_perturbations:
        {state["attention_perturbations"]}

        Cross Connections:
        {state["cross_connections"]}
        
        The exploration phase is complete.
        
        Now converge the reasoning.
        
        Identify the most promising ideas, connections, and cross-connections
        
        Consider:
        - relevance to the problem
        - potential usefulness
        - originality
        - feasibility
        - strength of the underlying reasoning
        
        Do not generate completely new ideas.
        Select and combine the strongest existing reasoning.
        
        Return ONLY valid JSON in exactly this format:
        
        {{
            "solutions": [
                "solution 1",
                "solution 2",
                "solution 3",
                "solution 4",
                "solution 5"
            ]
        }}
    """

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=400,
        temperature=0.4,
        do_sample=True
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    result = json.loads(response)

    return result["solutions"]

In [37]:
def final_output(state, max_rounds=3):

    for round_number in range(max_rounds):

        state["round"] = round_number + 1

        print(f"\n========== ROUND {state['round']} ==========")

        # 1. Divergence
        diverge(state)

        # 2. Parallel exploration
        associative_search(state)
        perspective_switch(state)
        attention_perturb(state)

        # 3. Connect everything
        cross_connect(state)

        # 4. Laya selects promising candidates
        laya_select(state)

        # 5. Evaluate diversity / coverage
        evaluation = evaluate_similar_or_not(state)

        print("\nEvaluation:")
        print(evaluation)

        # 6. If exploration is sufficient, stop
        if not evaluation["still_similar"]:
            print("\nExploration is sufficiently diverse.")
            break

        # 7. Otherwise, use missing directions
        #    to guide the next divergence round
        print("\nExploration is still too similar.")
        print("Missing directions:")
        print(state["missing_directions"])

    return state["selected_candidates"]

In [38]:
result = final_output(state)
print(result)


========== ROUND 1 ==========
RAW RESPONSE:
{
    "ideas": [
        "Create a targeted Instagram ad campaign using Facebook Ads Manager to reach users interested in your niche, with a focus on location-based targeting and high-intent audiences.",
        "Leverage Instagram Stories with polls, quizzes, and interactive questions to engage your audience and encourage shares, which can help increase visibility and follower growth.",
        "Collaborate with micro-influencers in your niche who have a loyal following and can promote your page to their audience in exchange for free products or services.",
        "Use Instagram Reels to create short, engaging, and trending videos that align with your brand or content, increasing the chances of going viral and attracting new followers.",
        "Run a giveaway contest on your Instagram page, encouraging followers to tag friends, share your post, and follow your account to enter, which can boost engagement and follower count.",
        "Ut

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/laya/router.py:456: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  agent = Agent(repo, **kwargs)



Evaluation:
{'still_similar': True, 'missing_directions': ['Content creation strategies (e.g., video content, storytelling, behind-the-scenes)', 'Engagement tactics (e.g., contests, polls, Q&A sessions)', 'Collaborations and partnerships (e.g., influencer collaborations, brand partnerships)']}

Exploration is still too similar.
Missing directions:
['Content creation strategies (e.g., video content, storytelling, behind-the-scenes)', 'Engagement tactics (e.g., contests, polls, Q&A sessions)', 'Collaborations and partnerships (e.g., influencer collaborations, brand partnerships)']

========== ROUND 2 ==========
RAW RESPONSE:
{
  "ideas": [
    "Launch a 7-day Instagram challenge with a unique hashtag, encouraging followers to share their own content related to the theme.",
    "Host a live Q&A session with an industry expert or influencer to generate buzz and engagement around the account.",
    "Create a series of short, visually striking stories that highlight the journey of growing t